In [173]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

In [174]:
weather_data = (pd.read_csv("./dataset/Weather_Data.csv"))[["Sunshine", "Evaporation"]]

In [175]:
weather_data = weather_data.head(100)

# Part 1 : Descriptive Statistics

## Part 1.1 : Dispersion

In [176]:
weather_data.describe()

,Sunshine,Evaporation
count,100.000,100.000000
mean,7.073,4.863000
std,3.763,1.906748
min,0.000,1.200000
25%,3.300,3.400000
50%,8.500,4.600000
75%,10.325,6.400000
max,12.200,10.600000


In [177]:
# Create new dataset with intermediate calculation columns
new_data = weather_data.copy()
new_data['Observation'] = range(1, len(new_data) + 1)

# Compute means
mean_sunshine = new_data['Sunshine'].mean()
mean_evaporation = new_data['Evaporation'].mean()

# Add deviation and power columns for Sunshine
new_data['Sunshine - Mean_S'] = new_data['Sunshine'] - mean_sunshine
new_data['(Sunshine - Mean_S)^2'] = new_data['Sunshine - Mean_S'] ** 2

# Add deviation and power columns for Evaporation
new_data['Evaporation - Mean_E'] = new_data['Evaporation'] - mean_evaporation
new_data['(Evaporation - Mean_E)^2'] = new_data['Evaporation - Mean_E'] ** 2

In [178]:
# Create separate tables for Sunshine and Evaporation
sunshine_data = new_data[['Observation', 'Sunshine', 'Sunshine - Mean_S', '(Sunshine - Mean_S)^2']].copy()
evaporation_data = new_data[['Observation', 'Evaporation', 'Evaporation - Mean_E', '(Evaporation - Mean_E)^2']].copy()

# Display the Sunshine table
print("Sunshine Data Table:")
sunshine_data.head()

Sunshine Data Table:


,Observation,Sunshine,Sunshine - Mean_S,(Sunshine - Mean_S)^2
0,1,0.0,-7.073,50.027329
1,2,2.7,-4.373,19.123129
2,3,0.1,-6.973,48.622729
3,4,0.0,-7.073,50.027329
4,5,0.0,-7.073,50.027329


In [179]:
# Display the Evaporation table
print("Evaporation Data Table:")
evaporation_data.head()

Evaporation Data Table:


,Observation,Evaporation,Evaporation - Mean_E,(Evaporation - Mean_E)^2
0,1,6.2,1.337,1.787569
1,2,3.4,-1.463,2.140369
2,3,2.4,-2.463,6.066369
3,4,2.2,-2.663,7.091569
4,5,4.8,-0.063,0.003969


In [180]:
import numpy as np

# Add class interval to Sunshine data table
min_s = sunshine_data['Sunshine'].min()
max_s = sunshine_data['Sunshine'].max()
bin_width = 3
# Calculate number of bins needed to cover all data
num_bins = int(np.ceil((max_s - min_s) / bin_width))
# Create bins starting from a value slightly below min to include all points
bin_start = np.floor(min_s / bin_width) * bin_width
bins_s = [bin_start + i * bin_width for i in range(num_bins + 1)]
sunshine_data['Class Interval'] = pd.cut(sunshine_data['Sunshine'], bins=bins_s, right=False)

# Create Evaporation Frequency Distribution Table
freq_s = sunshine_data["Class Interval"].value_counts().sort_index()
# Store original intervals for midpoint calculation
original_intervals_s = freq_s.index

sunshine_freq_table = pd.DataFrame({
    'Class Interval': [f"{interval.left} - {interval.right}" for interval in freq_s.index],
    'Frequency': freq_s.values
})

# Add extra columns
sunshine_freq_table["Mid of the interval (x)"] = [
    (interval.left + interval.right) / 2 for interval in original_intervals_s
]
# Use mid of modal class as assumed mean
modal_mid_s = sunshine_freq_table.loc[sunshine_freq_table['Frequency'].idxmax(), 'Mid of the interval (x)']
assumed_mean_s = modal_mid_s
h = 3
sunshine_freq_table['Deviation (d)'] = (sunshine_freq_table['Mid of the interval (x)'] - assumed_mean_s) / h
sunshine_freq_table['Deviation * Frequency'] = sunshine_freq_table['Deviation (d)'] * sunshine_freq_table['Frequency']
sunshine_freq_table['Deviation^2 * Frequency'] = sunshine_freq_table['Deviation (d)'] ** 2 * sunshine_freq_table['Frequency']

# Add totals row
total_freq_s = sunshine_freq_table['Frequency'].sum()
mean_from_grouped_s = assumed_mean_s + (sunshine_freq_table['Deviation * Frequency'].sum() / total_freq_s) * h
totals_s = pd.Series({
    'Class Interval': 'Total',
    'Frequency': total_freq_s,
    'Mid of the interval (x)': np.nan,
    'Deviation (d)': np.nan,
    'Deviation * Frequency': sunshine_freq_table['Deviation * Frequency'].sum(),
    'Deviation^2 * Frequency': sunshine_freq_table['Deviation^2 * Frequency'].sum()
})
sunshine_freq_table = pd.concat([sunshine_freq_table, pd.DataFrame([totals_s])], ignore_index=True)

# Display Sunshine Frequency Table
print("Sunshine Frequency Distribution Table:")
sunshine_freq_table

Sunshine Frequency Distribution Table:


,Class Interval,Frequency,Mid of the interval (x),Deviation (d),Deviation * Frequency,Deviation^2 * Frequency
0,0.0 - 3.0,21,1.5,-3.0,-63.0,189.0
1,3.0 - 6.0,14,4.5,-2.0,-28.0,56.0
2,6.0 - 9.0,22,7.5,-1.0,-22.0,22.0
3,9.0 - 12.0,41,10.5,0.0,0.0,0.0
4,12.0 - 15.0,2,13.5,1.0,2.0,2.0
5,Total,100,NaN,NaN,-111.0,269.0


In [181]:
# Add class interval to Evaporation data table
min_e = evaporation_data['Evaporation'].min()
max_e = evaporation_data['Evaporation'].max()
# Calculate number of bins needed to cover all data
num_bins_e = int(np.ceil((max_e - min_e) / bin_width))
# Create bins starting from a value slightly below min to include all points
bin_start_e = np.floor(min_e / bin_width) * bin_width
bins_e = [bin_start_e + i * bin_width for i in range(num_bins_e + 1)]
evaporation_data['Class Interval'] = pd.cut(evaporation_data['Evaporation'], bins=bins_e, right=False)

# Create Evaporation Frequency Distribution Table
freq_e = evaporation_data['Class Interval'].value_counts().sort_index()
# Store original intervals for midpoint calculation
original_intervals_e = freq_e.index

evaporation_freq_table = pd.DataFrame({
    'Class Interval': [f"{interval.left} - {interval.right}" for interval in freq_e.index],
    'Frequency': freq_e.values
})

# Add extra columns
evaporation_freq_table['Mid of the interval (x)'] = [(interval.left + interval.right) / 2 for interval in original_intervals_e]
# Use mid of modal class as assumed mean
modal_mid_e = evaporation_freq_table.loc[evaporation_freq_table['Frequency'].idxmax(), 'Mid of the interval (x)']
assumed_mean_e = modal_mid_e
evaporation_freq_table['Deviation (d)'] = (evaporation_freq_table['Mid of the interval (x)'] - assumed_mean_e) / h
evaporation_freq_table['Deviation * Frequency'] = evaporation_freq_table['Deviation (d)'] * evaporation_freq_table['Frequency']
evaporation_freq_table['Deviation^2 * Frequency'] = evaporation_freq_table['Deviation (d)'] ** 2 * evaporation_freq_table['Frequency']

# Add totals row
total_freq_e = evaporation_freq_table['Frequency'].sum()
mean_from_grouped_e = assumed_mean_e + (evaporation_freq_table['Deviation * Frequency'].sum() / total_freq_e) * h
totals_e = pd.Series({
    'Class Interval': 'Total',
    'Frequency': total_freq_e,
    'Mid of the interval (x)': np.nan,
    'Deviation (d)': np.nan,
    'Deviation * Frequency': evaporation_freq_table['Deviation * Frequency'].sum(),
    'Deviation^2 * Frequency': evaporation_freq_table['Deviation^2 * Frequency'].sum()
})
evaporation_freq_table = pd.concat([evaporation_freq_table, pd.DataFrame([totals_e])], ignore_index=True)

# Display Evaporation Frequency Table
print("Evaporation Frequency Distribution Table:")
evaporation_freq_table

Evaporation Frequency Distribution Table:


,Class Interval,Frequency,Mid of the interval (x),Deviation (d),Deviation * Frequency,Deviation^2 * Frequency
0,0.0 - 3.0,16,1.5,-1.0,-16.0,16.0
1,3.0 - 6.0,50,4.5,0.0,0.0,0.0
2,6.0 - 9.0,33,7.5,1.0,33.0,33.0
3,9.0 - 12.0,1,10.5,2.0,2.0,4.0
4,Total,100,NaN,NaN,19.0,53.0


In [182]:
# Export tables to excel files
sunshine_freq_table.to_excel("sunshine_frequency_distribution.xlsx", index=False)
evaporation_freq_table.to_excel("evaporation_frequency_distribution.xlsx", index=False)


## Part 1.2 : Central Tendency

In [189]:
# Calculate the mean, median, mode, standard deviation, and variance for both Sunshine and Evaporation datasets

# Sunshine statistics
mean_s = weather_data["Sunshine"].mean()
median_s = weather_data["Sunshine"].median()
mode_s = stats.mode(weather_data["Sunshine"]).mode
std_dev_s = weather_data["Sunshine"].std()
variance_s = weather_data["Sunshine"].var()

# Evaporation statistics
mean_e = weather_data["Evaporation"].mean()
median_e = weather_data["Evaporation"].median()
mode_e = stats.mode(weather_data["Evaporation"]).mode
std_dev_e = weather_data["Evaporation"].std()
variance_e = weather_data["Evaporation"].var()

print(
    f"""
Sunshine - 
Mean: {mean_s}, 
Median: {median_s}, 
Mode: {mode_s}, 
Std Dev: {std_dev_s}, 
Variance: {variance_s}
"""
)

print(
    f"""
Evaporation - 
Mean: {mean_e},
Median: {median_e},
Mode: {mode_e},
Std Dev: {std_dev_e},
Variance: {variance_e}
"""
)


Sunshine - 
Mean: 7.073, 
Median: 8.5, 
Mode: 0.0, 
Std Dev: 3.763000495252788, 
Variance: 14.160172727272728


Evaporation - 
Mean: 4.8629999999999995,
Median: 4.6,
Mode: 3.2,
Std Dev: 1.9067479851275255,
Variance: 3.635687878787878

